# Debugging PyTorch: the errors everyone hits, and a method to find them

_PyTorch (a complete course)_

**Most of your PyTorch time is debugging — so learn the greatest-hits errors, their fixes, and one method: overfit a single batch first.**

PyTorch errors feel random until you see they come from a small, fixed set of mismatches:
       shapes that do not line up, devices that disagree, dtypes that are wrong, or
       gradients that are missing, exploding, or never zeroed. Almost every message maps to one of these.

       So the cure is not memorizing messages &mdash; it is a routine that exposes the mismatch:
       set a seed (so the bug is reproducible), overfit a single batch (separates wiring bugs from data
       bugs), print shapes, dtypes, and devices at the failing line, and check gradient norms (zero means
       no learning signal; huge means it is about to explode). When the routine still hides the source, turn on
       set_detect_anomaly(True) and let PyTorch point at the operation.

---

This notebook is a practice scaffold for the **Debugging PyTorch: the errors everyone hits, and a method to find them** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)                       # STEP 1 of the method: make bugs reproducible
device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# A) REPRODUCE A SHAPE MISMATCH, THEN FIX IT
#    Symptom: "mat1 and mat2 shapes cannot be multiplied".
# ============================================================
x = torch.randn(16, 20)                    # batch of 16, 20 features each
bad_layer = nn.Linear(10, 4)               # WRONG: expects 10 in-features, gets 20
try:
    bad_layer(x)
except RuntimeError as e:
    print("CAUGHT shape error:", str(e).splitlines()[0])
    # FIX: print the shape, then size the layer to match.
    print("x.shape =", tuple(x.shape))     # -> (16, 20)  => in_features must be 20
good_layer = nn.Linear(20, 4)              # in_features now matches x's last dim
print("fixed output shape:", tuple(good_layer(x).shape))   # -> (16, 4)

# ============================================================
# B) A TINY MODEL + THE OVERFIT-ONE-BATCH SANITY CHECK
#    If the loss can't reach ~0 on ONE batch, the bug is in
#    the model/loss, not the data or the learning rate.
# ============================================================
model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 3)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()            # expects LOGITS + LONG class indices

# one fixed batch (note: data must go to the SAME device as the model)
xb = torch.randn(16, 20, device=device)
yb = torch.randint(0, 3, (16,), device=device)   # class indices in [0, 2], dtype long

# Print shapes / dtypes / devices BEFORE the loop — the cheapest debug tool there is.
print("xb:", tuple(xb.shape), xb.dtype, xb.device)
print("yb:", tuple(yb.shape), yb.dtype, yb.device)   # long, same device

for step in range(200):
    optimizer.zero_grad()                  # clear grads — forgetting this is the #1 "loss won't move" bug
    logits = model(xb)                     # raw logits, shape (16, 3) — NO softmax before CrossEntropyLoss
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        # STEP 4: gradient-norm check. ~0 => no signal reaching weights; huge => about to explode.
        total_norm = sum(p.grad.norm().item() ** 2 for p in model.parameters()) ** 0.5
        print(f"step {step:3d}  loss {loss.item():.4f}  grad_norm {total_norm:.3f}")
print("overfit-one-batch final loss:", round(loss.item(), 5), "(should be ~0 -> wiring is correct)")

# ============================================================
# C) NaN-LOSS GUARD + ANOMALY DETECTION
#    Catch a NaN the instant it appears and find the op that made it.
# ============================================================
torch.autograd.set_detect_anomaly(True)    # slow; turn on only while hunting a NaN / in-place bug

def guarded_step(logits, target):
    loss = loss_fn(logits, target)
    if not torch.isfinite(loss):           # NaN or inf guard
        raise FloatingPointError(f"non-finite loss {loss.item()} — lower lr / clip grads / check inputs")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # guard against exploding grads
    return loss

optimizer.zero_grad()
loss = guarded_step(model(xb), yb)
optimizer.step()
print("guarded step ok, loss:", round(loss.item(), 4))

# ============================================================
# D) THE SILENT BROADCASTING BUG (no error, wrong answer)
# ============================================================
pred = torch.randn(4, 1)                    # shape (4, 1)
target = torch.randn(4)                     # shape (4,)
wrong = (pred - target)                      # broadcasts to (4, 4) — silently WRONG
right = (pred.squeeze(1) - target)           # shapes match -> (4,)
print("broadcast shapes -> wrong:", tuple(wrong.shape), " right:", tuple(right.shape))

# ============================================================
# E) IN-PLACE OP THAT BREAKS AUTOGRAD (and the fix)
# ============================================================
w = torch.randn(3, requires_grad=True)
try:
    w += 1                                   # in-place on a leaf that requires grad -> RuntimeError
except RuntimeError as e:
    print("CAUGHT in-place error:", str(e).splitlines()[0])
with torch.no_grad():                        # FIX: edit weights inside no_grad (or use out-of-place w = w + 1)
    w += 1
print("in-place fix ok, w mean:", round(w.mean().item(), 3))

## Visualize the data & results

_Forgetting optimizer.zero_grad() makes gradients accumulate across steps. What does that do to the training-loss curve, compared with the correctly-zeroed run?_

In [ ]:
import numpy as np

# Tiny linear regression: fit y = 2x + 1 by gradient descent.
# FIXED run zeroes the gradient each step; BUGGY run never does
# (it accumulates every past gradient — exactly what forgetting
#  optimizer.zero_grad() does in PyTorch).
rng = np.random.default_rng(0)
N = 64
x = rng.normal(0, 1, N)
y = 2.0 * x + 1.0 + rng.normal(0, 0.1, N)

def grads(w, b):
    err = (w * x + b) - y
    loss = np.mean(err ** 2)
    return loss, 2 * np.mean(err * x), 2 * np.mean(err)

lr, EPOCHS = 0.1, 40

# FIXED: fresh gradient each step
w, b, fixed = 0.0, 0.0, []
for _ in range(EPOCHS):
    loss, gw, gb = grads(w, b); fixed.append(loss)
    w -= lr * gw; b -= lr * gb

# BUGGY: gradients accumulate (zero_grad forgotten)
w, b, accw, accb, buggy = 0.0, 0.0, 0.0, 0.0, []
for _ in range(EPOCHS):
    loss, gw, gb = grads(w, b); buggy.append(loss)
    accw += gw; accb += gb            # never cleared
    w -= lr * accw; b -= lr * accb

idx = [0,2,5,7,10,13,15,18,20,23,26,28,31,33,36,39]
print("fixed:", [(i, round(fixed[i], 3)) for i in idx])
print("buggy:", [(i, round(buggy[i], 3)) for i in idx])

import matplotlib.pyplot as plt
plt.plot(buggy, color="#ff7b72", label="buggy (grads accumulate)")
plt.plot(fixed, color="#7ee787", label="fixed (zero_grad each step)")
plt.xlabel("epoch"); plt.ylabel("mean squared error")
plt.title("Buggy (forgot zero_grad) vs fixed training loss")
plt.legend(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your training loss bounces around and never settles &mdash; some epochs it drops, then it jumps back up, with no steady descent. You did call loss.backward() and optimizer.step() every iteration. What is the single most likely cause, and how do you confirm and fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that PyTorch accumulates gradients into .grad by default; they are not cleared automatically. — _If you never clear them, each step's gradient is added on top of every previous step's, so the update direction and size grow erratically._
- Check whether optimizer.zero_grad() runs at the top of every iteration, before backward(). — _Missing or misplaced zero_grad is the textbook cause of a loss that refuses to descend smoothly._
- Add optimizer.zero_grad() as the first line of the loop body and re-run. — _Now each step uses only the current batch's gradient, so updates are the right size and the loss descends._

**Answer:** You almost certainly forgot optimizer.zero_grad(). PyTorch adds new gradients onto whatever is already in .grad, so without clearing them they pile up across steps and the optimizer takes wildly oversized, ever-changing steps &mdash; the loss oscillates instead of descending. Put optimizer.zero_grad() at the start of each loop iteration (before loss.backward()). The CODEVIZ chart on this page shows exactly this: the buggy curve thrashes between ~0.4 and ~4.5, while the fixed curve slides smoothly to ~0.009.

</details>

**Problem 2.** You feed nn.CrossEntropyLoss the output of a final softmax layer, with one-hot encoded targets, and training barely moves. What two things are wrong, and what should you pass instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remember that nn.CrossEntropyLoss applies log_softmax internally. — _It expects raw logits; if you softmax first, you softmax twice, squashing the signal so gradients are tiny and learning crawls._
- Remember it expects integer class indices of shape (N,), not one-hot vectors. — _One-hot float targets are the wrong shape/type for the index-based loss and either error or compute the wrong thing._
- Remove the final softmax so the model outputs logits of shape (N, C), and pass targets as a long tensor of class indices in [0, C-1]. — _That is the exact input contract of CrossEntropyLoss; gradients flow correctly and the loss drops._

**Answer:** Two mistakes. (1) You softmaxed the outputs, but nn.CrossEntropyLoss already does log_softmax inside &mdash; so the network is softmaxed twice and the gradient nearly vanishes. (2) You passed one-hot targets, but it wants integer class indices. Fix both: have the model return raw logits of shape (N, C) (no final softmax), and pass targets as a torch.long tensor of shape (N,) holding the class index of each example.

</details>

**Problem 3.** On a Graphics Processing Unit (GPU), your first training step throws RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu. You already wrote model = model.to(device). Where is the cpu tensor, and what is the systematic fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the error: weights are on cuda:0, but some operand is still on cpu. — _A matmul or add needs both operands on the same device; one of them was never moved._
- Check the data path &mdash; the inputs and the targets coming off the DataLoader are on the CPU by default. — _Moving the model does not move the batches; each batch starts on the CPU and must be sent over explicitly._
- Inside the loop, do x = x.to(device) and y = y.to(device) for every batch (and confirm the loss's targets too). — _Now data and weights share a device, so every operation is legal._

**Answer:** The leftover CPU tensor is your data (and/or targets). Moving the model with .to(device) does not move the batches the DataLoader yields &mdash; those start on the CPU. The systematic fix is to choose one device at the top and move everything to it: model.to(device) once, and x = x.to(device), y = y.to(device) for every batch inside the loop.

</details>